In [ ]:
!pip install -q fastapi uvicorn python-docx google-genai pydantic requests
print("Installed")

In [ ]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Paste Gemini key: ")
print("Key saved")

In [ ]:
import os, json, re, time, uuid
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field
from google import genai
from docx import Document

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODELS = ["gemini-flash-latest", "gemini-flash-lite-latest"]  # primary, fallback
MAX_RETRIES = 3
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)


class AgentRequest(BaseModel):
    request: str = Field(..., min_length=10, max_length=2000)


# ---------- IMPROVEMENT: Retry + model fallback ----------
def call_llm(prompt, want_json, log, validate=None):
    last_err = None
    for model in MODELS:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                cfg = {"response_mime_type": "application/json"} if want_json else None
                resp = client.models.generate_content(
                    model=model, contents=prompt, config=cfg
                )
                text = (resp.text or "").strip()
                if not text:
                    raise ValueError("Empty response")
                if want_json:
                    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.M).strip()
                    result = json.loads(text)
                    if validate:
                        validate(result)
                    return result
                return text
            except Exception as e:
                last_err = e
                log.append(f"{model} attempt {attempt} failed: {type(e).__name__}")
                if attempt < MAX_RETRIES:
                    time.sleep(2 ** (attempt - 1))  # 1s, 2s backoff
        log.append(f"Switching away from {model}")
    raise RuntimeError(f"All models failed: {last_err}")


# ---------- Step 1: Agent makes its own plan ----------
def check_plan(plan):
    if not isinstance(plan, dict):
        raise ValueError("Plan is not an object")
    for key in ("doc_type", "title", "assumptions", "tasks"):
        if key not in plan:
            raise ValueError(f"Missing key: {key}")
    if not isinstance(plan["tasks"], list) or len(plan["tasks"]) < 2:
        raise ValueError("Too few tasks")
    for t in plan["tasks"]:
        if "title" not in t or "goal" not in t:
            raise ValueError("Bad task format")


def make_plan(user_request, log):
    prompt = f"""You are an autonomous business-document agent.
User request: "{user_request}"

1. Decide which document best fulfils the request (proposal, project plan,
   meeting minutes, business report, SOP, technical design, etc.).
2. If information is missing, vague or conflicting, make reasonable
   assumptions and list them.
3. Create your own TODO list of 4 to 6 tasks. Each task writes ONE section.

Return ONLY JSON in exactly this format:
{{"doc_type": "...", "title": "...", "assumptions": ["..."],
  "tasks": [{{"id": 1, "title": "section title", "goal": "what this section must contain"}}]}}"""
    return call_llm(prompt, True, log, validate=check_plan)


# ---------- Step 2: Agent executes each task ----------
def run_task(task, plan, user_request, written_titles, log):
    prompt = f"""You are writing one section of a "{plan['doc_type']}" titled "{plan['title']}".
Original user request: "{user_request}"
Assumptions made: {plan['assumptions']}
Sections already written: {written_titles}

Write the section "{task['title']}".
Goal: {task['goal']}

Rules: plain text only, no markdown headings, no bold marks.
Use short paragraphs. For lists, start each line with "- ".
Use realistic mock data where needed. Length: 120 to 200 words."""
    return call_llm(prompt, False, log)


# ---------- Step 3: Build the Word file ----------
def build_docx(plan, sections):
    doc = Document()
    doc.add_heading(plan["title"], 0)
    doc.add_paragraph(f"Document type: {plan['doc_type']}")
    if plan.get("assumptions"):
        doc.add_heading("Assumptions", 1)
        for a in plan["assumptions"]:
            doc.add_paragraph(str(a), style="List Bullet")
    for s in sections:
        doc.add_heading(s["title"], 1)
        for line in s["content"].split("\n"):
            line = line.strip()
            if not line:
                continue
            if line.startswith(("- ", "* ")):
                doc.add_paragraph(line[2:].replace("**", ""), style="List Bullet")
            else:
                doc.add_paragraph(line.lstrip("#").replace("**", "").strip())
    fname = f"{uuid.uuid4().hex[:8]}.docx"
    doc.save(os.path.join(OUT_DIR, fname))
    return fname


# ---------- API ----------
app = FastAPI(title="Autonomous Document Agent")


@app.post("/agent")
def agent(body: AgentRequest):
    log = []
    user_request = body.request.strip()

    try:
        plan = make_plan(user_request, log)
    except Exception as e:
        raise HTTPException(status_code=503, detail=f"Planning failed: {e}")

    sections, todo = [], []
    for task in plan["tasks"]:
        try:
            content = run_task(
                task, plan, user_request, [s["title"] for s in sections], log
            )
            status = "done"
        except Exception as e:
            content = "This section could not be generated. Please retry."
            status = "failed"
            log.append(f"Task '{task['title']}' failed: {e}")
        sections.append({"title": task["title"], "content": content})
        todo.append({"task": task["title"], "goal": task["goal"], "status": status})

    fname = build_docx(plan, sections)
    done = sum(1 for t in todo if t["status"] == "done")
    return {
        "message": f"Created '{plan['title']}' ({plan['doc_type']}). {done}/{len(todo)} tasks done.",
        "assumptions": plan["assumptions"],
        "todo_list": todo,
        "recovery_log": log,
        "download_url": f"/download/{fname}",
        "file_name": fname,
    }


@app.get("/download/{fname}")
def download(fname: str):
    if not re.fullmatch(r"[a-f0-9]{8}\.docx", fname):
        raise HTTPException(status_code=404, detail="File not found")
    path = os.path.join(OUT_DIR, fname)
    if not os.path.exists(path):
        raise HTTPException(status_code=404, detail="File not found")
    return FileResponse(path, filename=fname)


In [ ]:
import threading, time, uvicorn

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
print("Server running at http://127.0.0.1:8000")

In [ ]:
import requests, json
from google.colab import files


def ask(text):
    r = requests.post("http://127.0.0.1:8000/agent", json={"request": text}, timeout=600)
    data = r.json()
    print(json.dumps(data, indent=2))
    if "file_name" in data:
        files.download("outputs/" + data["file_name"])


print("Ready")

In [ ]:
# TEST 1: standard business request
ask("Create a project plan for launching a mobile app for a small coffee shop chain, including timeline, team roles, budget and risks.")

In [ ]:
# TEST 2: complex, vague request
ask("We need something for the board about the Q3 situation. Sales are down but also up in some regions, the CEO wants it short but also very detailed, and I do not have the numbers yet. Make it useful.")